# PYNQ-Z2 FPGA 边缘检测与车牌抓拍

1. 初始化 FPGA
2. ARM / FPGA 结果与耗时对比
3. 摄像头采集、ROI 保存与车牌识别

在 **PYNQ 板端 Jupyter** 从上往下逐格运行。先停止板端视频服务，避免摄像头和 DMA 被占用。

## 1. 参数与模块导入

`PC_IP` 填电脑地址，`CAMERA_INDEX` 填板端摄像头编号。其余路径自动从当前目录查找。

In [ ]:
from pathlib import Path
from datetime import datetime
from IPython.display import HTML, display
import base64
import html
import json
import sys
import time
import urllib.request

import cv2
import numpy as np

PC_IP = '192.168.2.1'
CAMERA_INDEX = 0

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent, cwd.parent.parent)
             if (p / 'edge_dma_runtime.py').is_file()), None)
assert ROOT is not None, '请从交付目录的 board_check/ 中打开本 Notebook'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from edge_dma_runtime import EdgeDmaRuntime
import opencv_benchmark as reference
import pynq_video_server as server

SAVE_DIR = ROOT / 'board_check' / 'results' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('项目目录：', ROOT)
print('图片保存目录：', SAVE_DIR)

In [ ]:
# 并排显示图像，不需要额外安装绘图库
def show_images(*images):
    blocks = []
    for title, image in images:
        ok, encoded = cv2.imencode('.png', image)
        assert ok, '图像编码失败'
        data = base64.b64encode(encoded).decode('ascii')
        blocks.append('<figure style="margin:8px;flex:1;min-width:200px">'
                      '<img style="width:100%" src="data:image/png;base64,' + data + '">'
                      '<figcaption>' + html.escape(title) + '</figcaption></figure>')
    display(HTML('<div style="display:flex;flex-wrap:wrap">' + ''.join(blocks) + '</div>'))

## 2. 加载 FPGA

加载交付的 `sobel_edge.bit/.hwh`，初始化 ROI 和边缘图两路 DMA。出现“已就绪”后继续。

In [ ]:
if globals().get('runtime') is not None:
    runtime.close()
runtime = EdgeDmaRuntime(ROOT, timeout_s=3.0)
runtime.print_status('FPGA 已就绪')

## 3. ARM / FPGA 对比

将三组标准图像送入 FPGA，并与 ARM OpenCV、预期结果对比。应看到六字段一致、边缘误差为 0。
六字段依次为：`x, y, w, h, 最大连通域网格数, 缩减率×100`。耗时取本次运行值。

In [ ]:
def compare_frame(gray):
    try:
        started = time.perf_counter()
        fields, edge, dma_ms = runtime.run_gray(gray)
        fpga_ms = (time.perf_counter() - started) * 1000
    except (Exception, KeyboardInterrupt):
        runtime.close()
        raise

    started = time.perf_counter()
    arm = reference.opencv_roi_pipeline(gray)
    arm_fields = reference.result_fields(arm)
    arm_ms = (time.perf_counter() - started) * 1000
    errors = int(np.count_nonzero(edge != arm['edge']))
    print('FPGA：', fields, ' ARM：', arm_fields, ' 边缘误差：', errors)
    print('FPGA 完整调用 {:.3f} ms | DMA 窗口 {:.3f} ms | ARM {:.3f} ms | ARM/FPGA {:.2f}x'
          .format(fpga_ms, dma_ms, arm_ms, arm_ms / fpga_ms))
    assert fields == arm_fields and errors == 0, 'ARM / FPGA 结果不一致'
    return fields, edge, arm['edge']

cases = [('single_target', [160, 160, 160, 160, 4, 9167]),
         ('multi_target', [80, 80, 240, 160, 6, 8750]),
         ('noise_target', [400, 160, 160, 160, 4, 9167])]

for name, expected_fields in cases:
    gray = np.fromfile(str(ROOT / 'validation' / (name + '_640x480_input.bin')),
                       dtype=np.uint8).reshape(480, 640)
    magnitude = np.fromfile(str(ROOT / 'validation' / (name + '_640x480_expected.bin')),
                            dtype=np.uint8).reshape(480, 640)
    expected_edge = np.where(magnitude >= 80, 255, 0).astype(np.uint8)
    print('\n---', name, '---')
    fields, edge, arm_edge = compare_frame(gray)
    assert fields == expected_fields and np.array_equal(edge, expected_edge), '与标准结果不一致'
    show_images(('输入图像', gray), ('FPGA 边缘', edge), ('ARM 边缘', arm_edge))

blank = np.zeros((480, 640), dtype=np.uint8)
fields, edge, _ = compare_frame(blank)
assert fields == [0, 0, 0, 0, 0, 10000] and not np.any(edge)
print('三组标准图像和空白帧均一致。')

## 4. 摄像头采集与同帧对比

把清晰车牌放入画面，运行下格。应显示彩色原图、ROI 框与 FPGA 边缘图。
没有有效 ROI 时，调整目标位置后重跑本格；要求摄像头输出原生 `640×480`。

In [ ]:
frame = None
roi_path = None
cap = cv2.VideoCapture(CAMERA_INDEX)
try:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    assert cap.isOpened(), '摄像头打不开，请检查 USB Host、编号及占用程序'
    for _ in range(5):
        ok, frame = cap.read()
        assert ok and frame is not None, '摄像头读帧失败'
finally:
    cap.release()

assert frame.shape == (480, 640, 3) and frame.dtype == np.uint8, '需要 640×480 BGR 图像'
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
fields, edge, _ = compare_frame(gray)
preview = frame.copy()
if server.valid_roi(fields):
    x, y, w, h = fields[:4]
    cv2.rectangle(preview, (x, y), (x + w - 1, y + h - 1), (0, 255, 0), 2)
else:
    print('当前没有有效 ROI，请调整目标后重跑本格。')
show_images(('摄像头与 ROI', preview), ('FPGA 边缘', edge))

## 5. 保存彩色 ROI

保存的是未画框的原始彩色裁剪图。查看下面图片，确认车牌完整、清晰。

In [ ]:
roi_path = None
assert frame is not None and server.valid_roi(fields), '先调整目标，重新运行第 4 步'
roi_path = server.save_roi(frame, fields, SAVE_DIR)
crop = cv2.imread(str(roi_path))
x, y, w, h = fields[:4]
assert crop is not None and crop.shape == (h, w, 3), '保存的 ROI 尺寸异常'
print('已保存：', roi_path)
show_images(('彩色 ROI', crop))

## 6. 连接电脑识别服务

先在电脑启动 `src/plate_lpr_server/start_server.cmd`（首次安装见 README）。
运行下格，正常应返回 `status: ok`。

In [ ]:
AI_BASE = 'http://{}:5000'.format(PC_IP)
http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
with http.open(AI_BASE + '/health', timeout=3.0) as response:
    health = json.loads(response.read().decode('utf-8'))
print(health)
assert health.get('status') == 'ok', '电脑识别服务未就绪'
assert health.get('auto_save_policy') == server.AUTO_CAPTURE_POLICY, '电脑与板端程序版本不匹配'

## 7. 识别当前 ROI

把刚保存的图片发给电脑。请将返回号码与实际车牌核对；未识别时调整清晰度，重跑第 4、5、7 步。

In [ ]:
assert roi_path is not None and roi_path.is_file(), '先完成第 5 步'
request = urllib.request.Request(
    AI_BASE + '/api/recognize', data=roi_path.read_bytes(),
    headers={'Content-Type': 'image/jpeg'}, method='POST')
started = time.perf_counter()
with http.open(request, timeout=3.0) as response:
    result = json.loads(response.read().decode('utf-8'))
print('接口返回：', result)
print('请求总耗时：{:.1f} ms'.format((time.perf_counter() - started) * 1000))
assert result.get('status') in ('success', 'fail'), '识别接口返回格式异常'
if result['status'] == 'success':
    print('车牌：', result['plate'], ' 置信分数：', result['confidence'])
else:
    print('当前图片未识别出车牌，请检查上面的 ROI 图片后重新采集。')

## 8. 释放硬件

运行下格后，保存 Notebook 并 **Shutdown Kernel**，再启动完整视频链路。
DMA 超时需要重启整块板子。

In [ ]:
if globals().get('runtime') is not None:
    runtime.close()
    runtime = None
print('下一步：关闭本内核，在板端终端运行以下命令。')
print('cd "{}"'.format(ROOT))
print('sudo -E python3 pynq_video_server.py --pc-ip {} --camera {}'.format(PC_IP, CAMERA_INDEX))

## 9. 运行完整视频链路

板端终端运行上格打印的命令，出现“等待电脑连接”后，在电脑 **CMD** 执行：

```bat
cd /d "D:\fpga_lpr\src\fpga_edge_vision\software\pc"
venv\Scripts\python.exe pc_dashboard.py --host 192.168.2.99
```

将 `--host` 改为板子实际 IP，点击“连接 PYNQ”。先看双画面，再试“抓拍当前 ROI”“ARM 对比”和自动抓拍开关。
自动抓拍稳定约 1 秒后请求识别，确认识别后才保存；每次完成后冷却约 4 秒。图片默认保存在板端项目目录。

结束时关闭电脑窗口，在板端服务终端按 `Ctrl+C`。